# 00 — Setup Google Colab & Google Drive

task:
1. Melakukan *mount* Google Drive (`/content/drive`).
2. Memverifikasi keberadaan direktori proyek `LAB_TA_GradCAM_Pneumonia` di `MyDrive`.
3. Memastikan seluruh subfolder keluaran (`data/interim`, `data/processed`, `models/unet_frozen`, `outputs/...`) tersedia.
4. Memeriksa ketersediaan GPU (T4 / V100 / A100) dan spesifikasi perangkat keras.
5. Menginstal pustaka pendukung tambahan (`requirements-extra.txt`) tanpa menginstal ulang PyTorch bawaan Colab.
6. Memverifikasi ketersediaan berkas dataset di `data/raw/kermany` dan `data/raw/rsua`.

In [ ]:
# Sel 1: Mount Google Drive ke /content/drive
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    print("Google Drive berhasil di-mount di /content/drive")
else:
    print("Notebook dijalankan di luar Colab (lingkungan lokal).")

### Analisis Sel 1
Google Drive dihubungkan ke `/content/drive`. Folder proyek Anda di Google Drive dapat diakses pada jalur `/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia`.

In [ ]:
# Sel 2: Verifikasi direktori PROJECT_ROOT dan inisialisasi folder keluaran
PROJECT_ROOT = Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia") if IN_COLAB else Path("./local_mirror").resolve()

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
assert PROJECT_ROOT.exists(), f"Direktori {PROJECT_ROOT} tidak ditemukan! Pastikan nama folder di MyDrive sesuai."

# Buat seluruh struktur subdirektori keluaran secara otomatis jika belum ada
dirs_to_create = [
    PROJECT_ROOT / "code",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "data" / "interim" / "harmonized_png",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "raw" / "rsna",
    PROJECT_ROOT / "data" / "raw" / "kermany_manual_masks",
    PROJECT_ROOT / "data" / "raw" / "seg_public",
    PROJECT_ROOT / "models" / "unet_frozen",
    PROJECT_ROOT / "outputs" / "checkpoints",
    PROJECT_ROOT / "outputs" / "logs",
    PROJECT_ROOT / "outputs" / "predictions",
    PROJECT_ROOT / "outputs" / "gradcam",
    PROJECT_ROOT / "outputs" / "metrics",
    PROJECT_ROOT / "outputs" / "stats",
    PROJECT_ROOT / "outputs" / "figures" / "eda",
    PROJECT_ROOT / "outputs" / "figures" / "audit",
    PROJECT_ROOT / "outputs" / "tables" / "eda",
    PROJECT_ROOT / "outputs" / "tables" / "audit",
]

for d in dirs_to_create:
    d.mkdir(parents=True, exist_ok=True)

print("Seluruh struktur direktori keluaran di Google Drive siap dan terverifikasi!")

### Analisis Sel 2
Seluruh folder kerja dan penampung artefak hasil eksperimen telah dibuat di Google Drive Anda tanpa menyentuh data mentah (*read-only*).

In [ ]:
# Sel 3: Pengecekan GPU dan versi perangkat lunak dasar
import torch
import torchvision

print(f"Python Version      : {sys.version.split()[0]}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"Torchvision Version : {torchvision.__version__}")
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Nama GPU            : {torch.cuda.get_device_name(0)}")
    print(f"Memori GPU          : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[PERINGATAN] GPU tidak aktif! Aktifkan GPU via Runtime -> Change runtime type -> T4 GPU")

### Analisis Sel 3
Memastikan PyTorch bawaan Colab mendeteksi kartu grafis GPU dengan baik untuk pelatihan model pada tahap berikutnya.

In [ ]:
# Sel 4: Instalasi dependensi tambahan (sesuai requirements-extra.txt) tanpa mengubah PyTorch bawaan
if IN_COLAB:
    !pip install -q timm==1.0.15 scikit-learn==1.6.1 scipy==1.15.2 pandas==2.2.3 pyarrow==19.0.1 opencv-python-headless==4.11.0.86 pyyaml==6.0.2 pydicom
    print("Dependensi tambahan berhasil dipasang!")
else:
    print("Mode lokal: gunakan virtual environment yang telah disediakan.")

### Analisis Sel 4
Pustaka tambahan yang diperlukan untuk `timm` (EfficientNet-B0), statistik, dan manipulasi data telah dipasang dengan versi ter-pin.

In [ ]:
# Sel 5: Verifikasi Ketersediaan Dataset Mentah di Google Drive
kermany_path = PROJECT_ROOT / "data" / "raw" / "kermany" / "chest_xray"
rsna_path = PROJECT_ROOT / "data" / "raw" / "rsna"
rsua_path = PROJECT_ROOT / "data" / "raw" / "rsua" / "Data Chest X-Ray RSUA (Validated)-20230618T025557Z-001" / "Data Chest X-Ray RSUA (Validated)"

print("=== VERIFIKASI DATASET RAW DI GOOGLE DRIVE ===")
if kermany_path.exists():
    n_train_norm = len(list((kermany_path / "train" / "NORMAL").glob("*.jpeg")))
    n_train_pneu = len(list((kermany_path / "train" / "PNEUMONIA").glob("*.jpeg")))
    n_test_norm = len(list((kermany_path / "test" / "NORMAL").glob("*.jpeg")))
    n_test_pneu = len(list((kermany_path / "test" / "PNEUMONIA").glob("*.jpeg")))
    total_kermany = n_train_norm + n_train_pneu + n_test_norm + n_test_pneu
    print(f"[OK] Kermany CXR Terdeteksi: {total_kermany} citra")
    print(f"     - train/NORMAL    : {n_train_norm} (target: 1349)")
    print(f"     - train/PNEUMONIA : {n_train_pneu} (target: 3883)")
    print(f"     - test/NORMAL     : {n_test_norm} (target: 234)")
    print(f"     - test/PNEUMONIA  : {n_test_pneu} (target: 390)")
else:
    print(f"[ERROR] Folder Kermany tidak ditemukan di: {kermany_path}")

rsna_labels = rsna_path / "stage_2_train_labels.csv"
rsna_cls = rsna_path / "stage_2_detailed_class_info.csv"
rsna_imgs = rsna_path / "stage_2_train_images"
if rsna_labels.exists() and rsna_cls.exists() and rsna_imgs.exists():
    n_dcm = len(list(rsna_imgs.glob("*.dcm"))) + len(list(rsna_imgs.glob("*.png")))
    print(f"[OK] RSNA Pneumonia Terdeteksi: {n_dcm} berkas citra (target penuh: 26684)")
else:
    print(f"[INFO] Dataset RSNA belum diletakkan di: {rsna_path}")
    print("       Jalankan Sel 6 (opsional) di bawah untuk mengunduh RSNA langsung dari Kaggle ke Drive.")

if rsua_path.exists():
    n_rsua_norm = len(list((rsua_path / "Non_Covid" / "images").glob("*.bmp")))
    n_rsua_pneu = len(list((rsua_path / "Non_Covid_Pneumonia" / "images").glob("*.bmp")))
    n_rsua_covid = len(list((rsua_path / "Covid" / "images").glob("*.bmp")))
    print(f"[OK] RSUA Validated Terdeteksi: {n_rsua_norm + n_rsua_pneu} citra evaluasi (dan {n_rsua_covid} Covid excluded)")
    print(f"     - Non_Covid (Normal)       : {n_rsua_norm} (target: 32)")
    print(f"     - Non_Covid_Pneumonia (Pneu): {n_rsua_pneu} (target: 53)")
else:
    print(f"[ERROR] Folder RSUA Validated tidak ditemukan di: {rsua_path}")

print("\nPersiapan lingkungan Colab selesai 100%! Anda dapat melanjutkan ke notebook berikutnya.")

In [ ]:
# Sel 6 (OPSIONAL — Jalankan sekali saja jika data/raw/rsna belum diunduh dari Kaggle)
# Syarat: sudah Join & Accept Rules di https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/rules
# dan punya berkas kaggle.json (dari Kaggle -> Settings -> API -> Create New Token).
DOWNLOAD_RSNA_NOW = False  # Ubah menjadi True untuk mengunduh langsung di Colab

if DOWNLOAD_RSNA_NOW and IN_COLAB:
    from google.colab import files  # type: ignore
    import zipfile

    kaggle_json = Path("/root/.kaggle/kaggle.json")
    if not kaggle_json.exists():
        print("Unggah berkas kaggle.json Anda:")
        uploaded = files.upload()
        kaggle_json.parent.mkdir(parents=True, exist_ok=True)
        kaggle_json.write_bytes(uploaded["kaggle.json"])
        kaggle_json.chmod(0o600)

    work_rsna = Path("/content/work/rsna_dl")
    work_rsna.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/work/rsna_dl

    zip_path = work_rsna / "rsna-pneumonia-detection-challenge.zip"
    rsna_out = PROJECT_ROOT / "data" / "raw" / "rsna"
    rsna_out.mkdir(parents=True, exist_ok=True)

    # Ekstrak HANYA 3 berkas yang dibutuhkan (lewati stage_2_test_images karena tanpa label)
    needed_prefixes = (
        "stage_2_train_labels.csv",
        "stage_2_detailed_class_info.csv",
        "stage_2_train_images/",
    )
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [m for m in zf.namelist() if m.startswith(needed_prefixes)]
        print(f"Mengekstrak {len(members)} berkas penting ke {rsna_out} ...")
        zf.extractall(rsna_out, members=members)
    print("[OK] Unduhan & ekstraksi RSNA selesai! Ubah DOWNLOAD_RSNA_NOW = False untuk eksekusi berikutnya.")